## The Domain Finder: Official Google Places API

In [69]:
import requests
from difflib import SequenceMatcher
import re
import os
from dotenv import load_dotenv
import json

load_dotenv() # Loads the hidden .env file

True

In [70]:
# --- YOUR GOOGLE CLOUD API KEY ---
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_API_KEY")
SERPER_API_KEY = os.getenv("SERPER_API")

In [ ]:
def clean_company_name(name):
    """Strips legal jargon for better API matching."""
    clean = re.sub(r'\b(PRIVATE LIMITED|LIMITED|LTD|OPC|LLP|INC|PVT)\b', '', name, flags=re.IGNORECASE)
    return re.sub(r'[^a-zA-Z0-9\s]', '', clean).strip()

def test_google_places_new_api(company_name, state):
    print(f"--- Testing Places API (NEW) for: {company_name} ---")
    
    # The NEW 2026 Endpoint
    url = "https://places.googleapis.com/v1/places:searchText"
    
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": GOOGLE_API_KEY,
        # This "FieldMask" is mandatory in the new API to control costs
        "X-Goog-FieldMask": "places.displayName,places.formattedAddress,places.websiteUri,places.businessStatus"
    }
    
    cleaned_name = clean_company_name(company_name)
    payload = {
        "textQuery": f"{cleaned_name} {state}"
    }
    
    response = requests.post(url, json=payload, headers=headers)
    data = response.json()
    
    if "places" in data and len(data["places"]) > 0:
        place = data["places"][0]
        gmb_name = place["displayName"]["text"]
        
        similarity = SequenceMatcher(None, cleaned_name.lower().replace(" ", ""), gmb_name.lower().replace(" ", "")).ratio()
        
        print(f"✅ GMB Profile Found!")
        print(f"   Input Name:    {cleaned_name}")
        print(f"   GMB Reg. Name: {gmb_name}")
        print(f"   Match Score:   {similarity:.2f}")
        
        if similarity > 0.4:
            print("\n--- Verified GMB Details ---")
            print(f"Address: {place.get('formattedAddress', 'None')}")
            
            website = place.get('websiteUri')
            if website:
                print(f"Website: {website}")
            else:
                print("Website: Not listed on GMB.")
            
            return True, website
        else:
            print("\n⚠️ WARNING: Name match too low. Likely a false positive.")
            return False, None
    else:
        return print(f"❌ No GMB Profile Found or API Error: {data}")

In [26]:
url_meta = test_google_places_new_api("VASUSENA MEDIA", "Karnataka")

--- Testing Places API (NEW) for: VASUSENA MEDIA ---
✅ GMB Profile Found!
   Input Name:    VASUSENA MEDIA
   GMB Reg. Name: Eedina
   Match Score:   0.53

--- Verified GMB Details ---
Address: M2M MEDIA NETWORK LLP, 1496/4, 3rd Cross Rd, M.R Palya, Mariyappanapalya, Srirampura, Bengaluru, Karnataka 560021, India
Website: https://www.eedina.com/


In [28]:
url_meta[1]

'https://www.eedina.com/'

## DUCK DUCK GO..

In [65]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import urllib.parse
import re

# We bring in the clean name function to avoid triggering MCA directories
def clean_company_name(name):
    clean = re.sub(r'\b(PRIVATE LIMITED|LIMITED|LTD|OPC|LLP|INC|PVT)\b', '', name, flags=re.IGNORECASE)
    return re.sub(r'[^a-zA-Z0-9\s]', '', clean).strip()

def fallback_duckduckgo_search(company_name, state):
    print(f"--- Running Hardened DuckDuckGo Backup for: {company_name} ---")
    
    # 1. Clean the name to find the BRAND, not the legal entity
    cleaned_name = clean_company_name(company_name)
    query = f"{cleaned_name} {state} official website"
    url = f"https://html.duckduckgo.com/html/?q={query}"
    
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    }
    
    # 2. The Ultimate Indian B2B Blacklist
    B2B_BLACKLIST = [
        'zaubacorp', 'tofler', 'indiamart', 'justdial', 'dubleu', 'falconebiz',
        'linkedin', 'facebook', 'instagram', 'thecompanycheck', 'economictimes', 
        'instafinancials', 'ambitionbox', 'quickcompany', 'tradeindia', 'masterdata', 
        'zoominfo', 'glassdoor', 'owler', 'tracxn', 'crunchbase', 'pitchbook',
        'dir.indiamart', 'fundoodata', 'connect2india', 'vakilsearch', 'cleartax',
        'startupindia', 'easytoca', 'companydetails', '99corporates', 'fincrif', 'planetexim',
        'mastersindia', 'corpdir'
    ]
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print("DDG blocked the request. Try again later.")
            return "BLOCKED"
            
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 3. Loop through results and filter the garbage
        for a_tag in soup.find_all('a', class_='result__snippet'):
            link = a_tag.get('href')
            if link:
                parsed = urlparse(link)
                actual_url = parsed.query.split('uddg=')[1].split('&')[0] if 'uddg=' in parsed.query else link
                actual_url = urllib.parse.unquote(actual_url)

                # --- 🚨 THE AD BLOCKER 🚨 ---
                if 'duckduckgo.com/y.js' in actual_url or 'ad_domain' in actual_url:
                    print("   [Skipped Ad]: Sponsored Result")
                    continue
                
                domain = urlparse(actual_url).netloc.lower()
                
                # Check if the domain contains any of our blacklisted words
                if any(bad_word in domain for bad_word in B2B_BLACKLIST):
                    print(f"   [Skipped Aggregator]: {domain}")
                    continue # Skip to the next search result

                # --- 🚨 NEW: The Non-Commercial Shield 🚨 ---
                if domain.endswith(('.gov.in', '.gov', '.edu', '.edu.in', '.nic.in', 'karnataka.gov.in')):
                    print(f"   [Skipped Non-Commercial]: {domain}")
                    continue
                
                # --- 🚨 NEW: Root Domain Extractor 🚨 ---
                # This turns 'https://bundl.com/about-us' into 'https://bundl.com'
                parsed_clean = urlparse(actual_url)
                root_url = f"{parsed_clean.scheme}://{parsed_clean.netloc}"

                # --- 🚨 UPGRADED: Ghost Company Shield 🚨 ---
                domain_word = parsed_clean.netloc.replace('www.', '').split('.')[0].lower()
                comp_words = cleaned_name.lower().split()
                
                # 1. Check if any major word (>2 letters) from the company name is in the domain
                word_match = any(word in domain_word or domain_word in word for word in comp_words if len(word) > 2)
                
                # 2. Check if the domain is a legitimate acronym of the company name
                acronym = "".join([w[0] for w in comp_words if w])
                acronym_match = (domain_word == acronym) or (domain_word in acronym and len(domain_word) >= 2)
                
                if not (word_match or acronym_match):
                    print(f"   [Skipped Ghost/Unrelated]: {root_url}")
                    continue
                
                print(f"✅ Found Organic URL via DuckDuckGo: {root_url}")
                return root_url
                
        print("❌ No valid website found on DuckDuckGo after filtering.")
        return None
        
    except Exception as e:
        print(f"Scraper Error: {e}")
        return None

# Test the upgraded scraper
fallback_duckduckgo_search("VASUSENA MEDIA PRIVATE LIMITED", "Karnataka")

--- Running Hardened DuckDuckGo Backup for: VASUSENA MEDIA PRIVATE LIMITED ---
   [Skipped Aggregator]: www.thecompanycheck.com
   [Skipped Aggregator]: companies.dubleu.co
   [Skipped Aggregator]: www.falconebiz.com
   [Skipped Aggregator]: fincrif.com
   [Skipped Aggregator]: company.vakilsearch.com
   [Skipped Aggregator]: tracxn.com
   [Skipped Aggregator]: www.tofler.in
   [Skipped Aggregator]: www.instafinancials.com
   [Skipped Aggregator]: www.planetexim.net
   [Skipped Aggregator]: www.companydetails.in
❌ No valid website found on DuckDuckGo after filtering.


In [74]:
# fallback_duckduckgo_search("VASUSENA HOLDINGS AND INVESTMENTS PRIVATE LIMITED", "Karnataka")

fallback_duckduckgo_search("BUNDL TECHNOLOGIES", "Karnataka")

--- Running Hardened DuckDuckGo Backup for: BUNDL TECHNOLOGIES ---
   [Skipped Aggregator]: www.companydetails.in
   [Skipped Aggregator]: www.companydetails.in
✅ Found Organic URL via DuckDuckGo: https://www.bundl.com


'https://www.bundl.com'

## 3rd layer: SERPER.DEV

In [72]:
def fallback_serper_search(company_name, state):
    print(f"--- 🚨 TRIGGERING LAYER 3: Serper.dev for {company_name} ---")
    
    cleaned_name = clean_company_name(company_name)
    query = f"{cleaned_name} {state} official website"
    
    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query})
    headers = {
        'X-API-KEY': SERPER_API_KEY,
        'Content-Type': 'application/json'
    }
    
    try:
        response = requests.request("POST", url, headers=headers, data=payload)
        data = response.json()
        
        # If Google has no organic results at all, it's a true ghost
        if 'organic' not in data:
            print("❌ No organic results found on Google.")
            return None
            
        for result in data['organic']:
            actual_url = result.get('link')
            if not actual_url:
                continue
                
            parsed_clean = urlparse(actual_url)
            domain = parsed_clean.netloc.lower()
            root_url = f"{parsed_clean.scheme}://{parsed_clean.netloc}"
            
            # --- The Directory Shield ---
            if any(bad_word in domain for bad_word in B2B_BLACKLIST):
                print(f"   [L3 Skipped Aggregator]: {domain}")
                continue 
                
            # --- The Non-Commercial Shield ---
            if domain.endswith(('.gov.in', '.gov', '.edu', '.edu.in', '.nic.in', 'karnataka.gov.in')):
                print(f"   [L3 Skipped Non-Commercial]: {domain}")
                continue
                
            # --- The Ghost Company Shield ---
            domain_word = parsed_clean.netloc.replace('www.', '').split('.')[0].lower()
            comp_words = cleaned_name.lower().split()
            
            word_match = any(word in domain_word or domain_word in word for word in comp_words if len(word) > 2)
            acronym = "".join([w[0] for w in comp_words if w])
            acronym_match = (domain_word == acronym) or (domain_word in acronym and len(domain_word) >= 2)
            
            if not (word_match or acronym_match):
                print(f"   [L3 Skipped Ghost]: {root_url}")
                continue
            
            print(f"✅ Found Organic URL via Serper: {root_url}")
            return root_url
            
        print("❌ No valid website found on Serper after filtering.")
        return None
        
    except Exception as e:
        print(f"Layer 3 API Error: {e}")
        return None

In [73]:
fallback_serper_search("VASUSENA HOLDINGS AND INVESTMENTS PRIVATE LIMITED", "Karnataka")

--- 🚨 TRIGGERING LAYER 3: Serper.dev for VASUSENA HOLDINGS AND INVESTMENTS PRIVATE LIMITED ---
Layer 3 API Error: name 'B2B_BLACKLIST' is not defined


## SWAP: Parent to Child? 

In [75]:
# --- THE AD-SPEND PIVOT DICTIONARY ---
# Maps the corporate holding domain to the high-intent consumer domain
PARENT_TO_CHILD_DOMAIN = {
    # E-Commerce & Delivery
    "bundl.com": "swiggy.com",
    "one97.com": "paytm.com",
    "oravel.com": "oyorooms.com",
    "apiholdings.in": "pharmeasy.in",
    "brainbees.com": "firstcry.com",
    "fsnecommerce.com": "nykaa.com",
    "innovativeretail.in": "bigbasket.com",
    "fashnear.com": "meesho.com",
    "grofers.com": "blinkit.com",
    "zomato.com": "zomato.com", # Usually fine, but good to have
    
    # Fintech & SaaS
    "resilientinnovations.com": "bharatpe.com",
    "defmacro.in": "cleartax.in",
    "nextbillion.in": "groww.in",
    "dreamplug.com": "cred.club",
    "razorpay.com": "razorpay.com",
    
    # Mobility & Services
    "anitechnologies.com": "olacabs.com",
    "roppen.in": "rapido.bike",
    "urbanclap.com": "urbancompany.com",
    "girnarsoft.com": "cardekho.com",
    
    # EdTech & Gaming
    "thinkandlearn.in": "byjus.com",
    "sortinghat.in": "unacademy.com",
    "sporta.in": "dream11.com",
    "games24x7.com": "my11circle.com",
    
    # Health & Wellness
    "curefit.com": "cult.fit",
    "1mg.com": "tata1mg.com"
}

In [76]:
def upgrade_to_ad_domain(extracted_url):
    """Intercepts corporate domains and swaps them for ad-spending consumer domains."""
    if not extracted_url:
        return None
        
    # Clean the URL to just get the base domain (e.g., 'bundl.com')
    parsed = urlparse(extracted_url)
    base_domain = parsed.netloc.replace('www.', '').lower()
    
    # Check if the domain is in our pivot dictionary
    if base_domain in PARENT_TO_CHILD_DOMAIN:
        child_domain = PARENT_TO_CHILD_DOMAIN[base_domain]
        new_url = f"https://www.{child_domain}"
        print(f"🔄 SWAPPED: Corporate site '{base_domain}' upgraded to Ad Domain -> {new_url}")
        return new_url
        
    # If not in the dictionary, just return the original URL
    return extracted_url

upgrade_to_ad_domain('https://www.bundl.com')

🔄 SWAPPED: Corporate site 'bundl.com' upgraded to Ad Domain -> https://www.swiggy.com


'https://www.swiggy.com'

## VALIDATE URL LOGIC

In [77]:
# --- THE MASTER B2B BLACKLIST ---
B2B_BLACKLIST = [
    'zaubacorp', 'tofler', 'indiamart', 'justdial', 'dubleu', 'falconebiz',
    'linkedin', 'facebook', 'instagram', 'thecompanycheck', 'economictimes', 
    'instafinancials', 'ambitionbox', 'quickcompany', 'tradeindia', 'masterdata', 
    'zoominfo', 'glassdoor', 'owler', 'tracxn', 'crunchbase', 'pitchbook',
    'dir.indiamart', 'fundoodata', 'connect2india', 'vakilsearch', 'cleartax',
    'startupindia', 'easytoca', 'companydetails', '99corporates', 'fincrif', 
    'planetexim', 'mastersindia', 'corpdir', 'blogspot', 'wordpress'
]

def is_valid_b2b_domain(url):
    """Checks if a URL is a legitimate business domain and not a directory/social site."""
    if not url:
        return False
        
    domain = url.lower()
    
    # 1. Check against the Master Blacklist
    if any(bad_word in domain for bad_word in B2B_BLACKLIST):
        return False
        
    # 2. Check against Non-Commercial domains
    if domain.endswith(('.gov.in', '.gov', '.edu', '.edu.in', '.nic.in')):
        return False
        
    return True

True

In [ ]:
def master_domain_finder(company_name, state):
    """The 3-Layer Master Engine: Returns (URL, Source)"""
    print(f"\n--- Processing: {company_name} ---")

    # --- LAYER 1: Google Places API ---
    try:
        gmb_success, website = test_google_places_new_api(company_name, state)
    except Exception as e:
        print(f"   [GMB Error]: {e}")
        gmb_success, website = False, None
    
    if gmb_success and website:
        if is_valid_b2b_domain(website):
            print(f"🎯 WIN (Layer 1 - GMB): {website}")
            return upgrade_to_ad_domain(website), "Layer 1 - GMB"
        else:
            print(f"   [GMB Rejected]: {website} is a social/directory link.")
            
    # --- LAYER 2: DuckDuckGo Fallback ---
    print("⚠️ GMB Failed or no website. Triggering Layer 2 (DuckDuckGo)...")
    fallback_url = fallback_duckduckgo_search(company_name, state)
    source = "Layer 2 - DuckDuckGo"
    
    # --- LAYER 3: Serper.dev Fallback (SOS) ---
    if fallback_url == "BLOCKED":
        print("🚨 Layer 2 Failed (IP Block). Triggering Layer 3 (Serper.dev)...")
        fallback_url = fallback_serper_search(company_name, state)
        source = "Layer 3 - Serper"
    
    # --- FINAL VERDICT & SWAP ---
    if fallback_url and fallback_url != "BLOCKED":
        final_url = upgrade_to_ad_domain(fallback_url)
        print(f"🎯 WIN (Fallback): {final_url}")
        return final_url, source
    else:
        print("❌ FAILED: No valid B2B website found in any engine.")
        return None, "Failed / Not Found"

## The Intent Filter: The "No-API" Pixel Checker

In [24]:
import requests
from bs4 import BeautifulSoup

In [32]:
def test_pixel_checker(url):
    print(f"--- Testing Pixel Extraction for: {url} ---")
    if not url.startswith('http'):
        url = 'https://' + url
        
    try:
        # We use a standard browser User-Agent so websites don't block us as a bot
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        
        print(f"Status Code: {response.status_code} (200 means success)")
        html_content = response.text.lower()
        
        # Look for the exact footprints of Google Ads / Analytics
        has_gtag = 'googletagmanager.com/gtag/js' in html_content
        has_gtm = 'googletagmanager.com/gtm.js' in html_content
        has_aw = 'aw-' in html_content # Google Ads specific conversion ID
        
        print("Findings:")
        print(f" - Google Site Tag (gtag): {has_gtag}")
        print(f" - Tag Manager (GTM):    {has_gtm}")
        print(f" - Ads Conversion (AW-): {has_aw}")
        
        if has_gtag or has_gtm or has_aw:
            print("\n✅ VERDICT: Highly likely running Google Ads/Analytics.")
        else:
            print("\n❌ VERDICT: No obvious Google tracking found.")
            
    except Exception as e:
        print(f"Connection Failed: {e}")

In [36]:
# Test this on a known e-commerce brand or your own website
if is_valid_url is True:
    test_pixel_checker(url_meta[1])
elif is_valid_url is False:
    print("URL is not valid, check Places API or create a backup for edge cases")
else: 
    print("Error!")

--- Testing Pixel Extraction for: https://www.eedina.com/ ---
Status Code: 200 (200 means success)
Findings:
 - Google Site Tag (gtag): True
 - Tag Manager (GTM):    False
 - Ads Conversion (AW-): True

✅ VERDICT: Highly likely running Google Ads/Analytics.
